In [1]:
from typing import Dict, Hashable, Iterable, Tuple, Any, Optional
from sklearn.metrics import adjusted_mutual_info_score, normalized_mutual_info_score


Node = Hashable
Time = Hashable
Key = Tuple[Node, Time]


def dynamic_mi(
    gt: Dict[Key, Any],
    pred: Dict[Key, Any],
    keys: Optional[Iterable[Key]] = None,
    average_method: str = "arithmetic",
    ignore_missing: bool = True,
    normalisation: str = ["ami","nmi"]
) -> float:
    if keys is None:
        if ignore_missing:
            keys = set(gt.keys()) & set(pred.keys())
        else:
            keys = set(gt.keys()) | set(pred.keys())

    keys = list(keys)

    y_true = []
    y_pred = []
    for k in keys:
        if ignore_missing and (k not in gt or k not in pred):
            continue
        if k not in gt or k not in pred:
            raise KeyError(f"node {k} is missing in either gt or pred, but ignore_missing=False.")
        y_true.append(gt[k])
        y_pred.append(pred[k])

    if len(y_true) == 0:
        raise ValueError("no samples left after filtering missing nodes: please check if gt and pred are defined on the same (u,t) samples.")

    if normalisation == "ami":
        return float(adjusted_mutual_info_score(y_true, y_pred, average_method=average_method))
    elif normalisation == "nmi":
        return float(normalized_mutual_info_score(y_true, y_pred, average_method=average_method)) 
    else:
        return ("AttributeError")

In [10]:
from __future__ import annotations

from typing import Dict, Hashable, Tuple, Any, Optional, Literal
import pandas as pd

Node = Hashable
Time = Hashable
Key = Tuple[Node, Time]


def build_partition_from_csv(
    csv_path: str,
    *,
    source_col: str = "source",
    destination_col: str = "destination",
    timestamp_col: str = "timestamp",
    source_commu_col: str = "source_commu",
    destination_commu_col: str = "destination_commu",
    sep: str = ",",
    header: Optional[int] = "infer",          # "infer" | None | 0
    skip_first_row: bool = False,
    dtype_source: Any = int,
    dtype_destination: Any = int,
    dtype_timestamp: Any = int,
    dtype_commu: Any = int,
    on_conflict: Literal["keep_first", "keep_last", "error"] = "keep_last",
) -> Dict[Key, Any]:
    usecols = [source_col, destination_col, timestamp_col, source_commu_col, destination_commu_col]

    df = pd.read_csv(
        csv_path,
        sep=sep,
        header=header,
        usecols=usecols,
        skiprows=1 if skip_first_row else None,
    )

    df[source_col] = df[source_col].astype(dtype_source)
    df[destination_col] = df[destination_col].astype(dtype_destination)
    df[timestamp_col] = df[timestamp_col].astype(dtype_timestamp)
    df[source_commu_col] = df[source_commu_col].astype(dtype_commu)
    df[destination_commu_col] = df[destination_commu_col].astype(dtype_commu)

    part: Dict[Key, Any] = {}

    def _assign(k: Key, v: Any):
        if k not in part:
            part[k] = v
            return
        if part[k] == v:
            return
        if on_conflict == "keep_first":
            return
        if on_conflict == "keep_last":
            part[k] = v
            return
        raise ValueError(f"Conflict on {k}: existing={part[k]} new={v}")

    for s, d, t, cs, cd in df[[source_col, destination_col, timestamp_col, source_commu_col, destination_commu_col]].itertuples(index=False, name=None):
        _assign((s, t), cs)
        _assign((d, t), cd)

    return part


gt = build_partition_from_csv("data/syn_net.csv")
pred = build_partition_from_csv("result/tgn_community.csv")

In [11]:
dynamic_ami_score = dynamic_mi(gt, pred, normalisation="ami")
print("Dynamic AMI =", dynamic_ami_score)

Dynamic AMI = 0.0024311337998161133


In [ ]:
gt = build_partition_from_csv("data/syn_net.csv")
lago_pred = build_partition_from_csv("result/lago_community.csv")
lago_score = dynamic_mi(gt, lago_pred, normalisation="nmi")
print("LAGO Dynamic AMI =", lago_score)

LAGO Dynamic AMI = 0.40353710395468495
